In [1]:
import os
import pandas as pd
import sys
from pathlib import Path
import numpy as np
import natsort

project_root = Path.cwd().parents[0]
sys.path.append(str(project_root))

from algorithms.tss_file import TSSFile
from algorithms.wfm_edge import WFM_edge

In [2]:
folder = r'Z:\DynamicTestData\P42025\PUMP_UP_SPW'
output_folder = r'C:\Users\FeiyuanYang\OneDrive - Cambridge GaN Devices Ltd\Documents\CGD documents\Projects\P42025'
labels = {
    'Vg': 'ch1',
    'Vgi': 'ch2',
    'Vd': 'ch3',
    'MC': 'ch5',
}
files = os.listdir(folder)
tss_files = natsort.natsorted(files)
multifiles = [f for f in files if 'Multiframe' in f]
singlefiles = [f for f in files if 'Multiframe' not in f]

In [3]:
output = [['file name', 'Variant', 'Mark ID', 'Vd', 'Frequency (kHz)', 'dead time (ns)', 'Vg', 'Rgon', 'Rgoff', 'temperature (C)', 'data type', 
           'Pulse number', 'Vgi peak (start) (V)',  'Vgi flat (end) (V)',  'MC peak (start) (V)',  'MC flat (end) (V)',  
]]

In [4]:
for f in multifiles:
    names = f.split('_')
    tss = TSSFile(folder, f)
    time = tss.waveforms[labels['Vg']].time_for_frame()
    frame = tss.waveforms[labels['Vg']].frame_count
    rgon = names[6].split('R')[0]
    rgoff = names[6].split('R')[1]
    temp = 25 if names[7] == 'RT' else names[7][:-1]
    data_type = 'Multiframe'

    time_interval = time[1] - time[0]
    time_window = int(100 * 1e-9 / time_interval)
    for i in range(frame):
        vgi = tss.waveforms[labels['Vgi']].values_for_frame(i)
        vd = tss.waveforms[labels['Vd']].values_for_frame(i)
        mc = tss.waveforms[labels['MC']].values_for_frame(i)
        try:
            vd_edges = WFM_edge(vd, time, rising_edge_number=1, falling_edge_number=2, peak_width=1, peak_distance=10, sigma=10)
            vgi_edges = WFM_edge(vgi, time, rising_edge_number=2, falling_edge_number=1, peak_width=1, peak_distance=10, sigma=10)
            idx_vd_fall_1 = vd_edges.get_edge(1, "falling", 99)['thresh_idx']
            idx_vd_rise_1 = vd_edges.get_edge(1, "rising", 1)['thresh_idx']
            if vd_edges.falling_edge_number > 1:
                idx_vd_fall_2 = vd_edges.get_edge(2, "falling", 99)['thresh_idx']
            else:
                idx_vd_fall_2 = len(vd) - 1
            idx_vgi_fall_1 = vgi_edges.get_edge(1, "falling", 99)['thresh_idx']
            vgi_peak = np.max(vgi[idx_vd_fall_1:idx_vd_fall_1 + time_window])
            vgi_flat = np.mean(vgi[idx_vgi_fall_1 - time_window:idx_vgi_fall_1])
            mc_peak = np.max(mc[idx_vd_rise_1 - 10:idx_vd_rise_1 + time_window])
            mc_flat = np.mean(mc[idx_vd_fall_2 - time_window:idx_vd_fall_2 - int(time_window / 2)])

        except Exception as e:
            vgi_peak = 0
            vgi_flat = 0
            mc_peak = 0
            mc_flat = 0
            print(f"Error occurred while processing frame {i} in file {f}: {e}")


        line = [f, names[0], names[1], names[2][:-2], names[3][:-3], names[4][:-2], names[5][:-2], 
                rgon, rgoff, temp, data_type,
                i + 1, vgi_peak, vgi_flat, mc_peak, mc_flat]
        output.append(line)

    print(f)

KB25ENG_2630A000V6_400Vd_500kHz_100ns_15Vg_15R1Rg_RT_Multiframe.tss
KB25ENG_2630A000V6_400Vd_500kHz_100ns_15Vg_2R1Rg_RT_Multiframe.tss
KB25ENG_2630A000V6_400Vd_500kHz_100ns_15Vg_33R1Rg_RT_Multiframe.tss
KB25ENG_2630A000V6_400Vd_500kHz_400ns_15Vg_15R1Rg_RT_Multiframe.tss
KB25ENG_2630A000V6_400Vd_500kHz_400ns_15Vg_2R1Rg_RT_Multiframe.tss
KB25ENG_2630A000V6_400Vd_500kHz_400ns_15Vg_33R1Rg_RT_Multiframe.tss
KB25ENG_2630A000V6_400Vd_65kHz_100ns_15Vg_15R1Rg_RT_Multiframe.tss
KB25ENG_2630A000V6_400Vd_65kHz_100ns_15Vg_2R1Rg_RT_Multiframe.tss
KB25ENG_2630A000V6_400Vd_65kHz_100ns_15Vg_33R1Rg_RT_Multiframe.tss
KB25ENG_2630A000V6_400Vd_65kHz_400ns_15Vg_15R1Rg_RT_Multiframe.tss
KB25ENG_2630A000V6_400Vd_65kHz_400ns_15Vg_2R1Rg_RT_Multiframe.tss
KB25ENG_2630A000V6_400Vd_65kHz_400ns_15Vg_33R1Rg_RT_Multiframe.tss
KB25ENG_2630A000XA_400Vd_500kHz_100ns_15Vg_15R1Rg_RT_Multiframe.tss
KB25ENG_2630A000XA_400Vd_500kHz_100ns_15Vg_2R1Rg_RT_Multiframe.tss
KB25ENG_2630A000XA_400Vd_500kHz_100ns_15Vg_33R1Rg_RT_Multif

In [5]:
output_df = pd.DataFrame(output)
output_path = os.path.join(output_folder, 'P42025_BHDFN_pump_up_extracted_data.csv')
output_df.to_csv(output_path, index=False, header=False)